In [4]:
#Packages
import os
import pandas as pd
from pathlib import Path

# Directory
directory = r"C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2"

# Panel A - Absolute Threshold 30 degrees
Lead_labour_30 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_30_labour_unrest.csv"))
Lead_strike_30 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_30_strike.csv"))
Lead_protest_30 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_30_protest.csv"))

# Panel B - Absolute Threshold 27 & 32 degrees
# Cold Region
Lead_labour_27 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_27_labour_unrest.csv"))
Lead_strike_27 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_27_strike.csv"))
Lead_protest_27 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_27_protest.csv"))

# Hot Region
Lead_labour_32 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_32_labour_unrest.csv"))
Lead_strike_32 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_32_strike.csv"))
Lead_protest_32 = pd.read_csv(os.path.join(directory, "Raw", "Lead_abs_32_protest.csv"))

def clean_df_lead(df):
    # Rename columns
    df = df.rename(columns={
        "Unnamed: 0": "Variable",
        ".": "est",
        "Unnamed: 2": "se"
    })
    
    # Keep only the second and third rows (index 2 and 3)
    df = df.iloc[2:4].reset_index(drop=True)
    return df

Lead_labour_30 = clean_df_lead(Lead_labour_30)
Lead_strike_30 = clean_df_lead(Lead_strike_30)
Lead_protest_30 = clean_df_lead(Lead_protest_30)

Lead_labour_27 = clean_df_lead(Lead_labour_27)
Lead_strike_27 = clean_df_lead(Lead_strike_27)
Lead_protest_27 = clean_df_lead(Lead_protest_27)

Lead_labour_32 = clean_df_lead(Lead_labour_32)
Lead_strike_32 = clean_df_lead(Lead_strike_32)
Lead_protest_32 = clean_df_lead(Lead_protest_32)

# Baseline #Obs you gave earlier
obs_baseline = {
    "Labour Conflict": 117024,
    "Protests": 85744,
    "Strikes": 71024,
}

obs_clim1 = {
    "Labour Conflict": 69552,
    "Protests": 55568,
    "Strikes": 40112
}

obs_clim2 = {
    "Labour Conflict": 47472,
    "Protests": 30176,
    "Strikes": 30912
}

# Building Table
baseline_dfs = {
    "Labour Conflict": Lead_labour_30,
    "Protests": Lead_protest_30,
    "Strikes": Lead_strike_30,
}

dv_order = ["Labour Conflict", "Protests", "Strikes"]

def starify_from_t(t):
    """Return LaTeX stars from a t-stat."""
    t_abs = abs(t)
    if t_abs >= 2.58:
        return r"\sym{***}"
    elif t_abs >= 1.96:
        return r"\sym{**}"
    elif t_abs >= 1.64:
        return r"\sym{*}"
    else:
        return ""

def fmt_coef(est, se):
    """coef (3dp) + stars, using t = est/se."""
    t = est / se if se != 0 else 0
    return f"{est:.3f}{starify_from_t(t)}"

def fmt_se(se):
    """Standard error in parentheses, 3dp."""
    return f"({se:.3f})"

lines = []

lines.append(r"\def\sym#1{\ifmmode^{#1}\else\(^{#1}\)\fi}")
lines.append(r"\begin{tabular}{l*{6}{c}}")
lines.append(r"\toprule")
lines.append(
    r"                    &\multicolumn{2}{c}{(1)}&\multicolumn{2}{c}{(2)}&\multicolumn{2}{c}{(3)}\\"
)
lines.append(
    r"                    &\multicolumn{2}{c}{Labour Conflict}&\multicolumn{2}{c}{Protests}&\multicolumn{2}{c}{Strikes}\\"
)
lines.append(r"\midrule")

# =======================
# Panel A: baseline
# =======================
lines.append(r"\multicolumn{7}{l}{\textbf{Panel A:} All Summer Days}\\")

# ----- Heat Day (previous 30 days) -----
coef_cells_prev = []
se_cells_prev = []

for dv in dv_order:
    df = baseline_dfs[dv]
    est = float(df.loc[0, "est"])   # row 0 = previous 30 days
    se = float(df.loc[0, "se"])
    coef_cells_prev.append(rf"\multicolumn{{2}}{{c}}{{{fmt_coef(est, se)}}}")
    se_cells_prev.append(rf"\multicolumn{{2}}{{c}}{{{fmt_se(se)}}}")

lines.append(r"Heat Day $\geq 30^{\circ}$C (previous 30 days) &" + " & ".join(coef_cells_prev) + r"\\")
lines.append(r"                           &" + " & ".join(se_cells_prev) + r"\\")

# ----- Heat Day (next 30 days) -----
coef_cells_next = []
se_cells_next = []

for dv in dv_order:
    df = baseline_dfs[dv]
    est = float(df.loc[1, "est"])   # row 1 = next 30 days
    se = float(df.loc[1, "se"])
    coef_cells_next.append(rf"\multicolumn{{2}}{{c}}{{{fmt_coef(est, se)}}}")
    se_cells_next.append(rf"\multicolumn{{2}}{{c}}{{{fmt_se(se)}}}")

lines.append(r"Heat Day $\geq 30^{\circ}$C (next 30 days)     &" + " & ".join(coef_cells_next) + r"\\")
lines.append(r"                           &" + " & ".join(se_cells_next) + r"\\")

# Panel A # Obs row
lines.append(r"\midrule")
obs_cells_A = []
for dv in dv_order:
    obs_cells_A.append(rf"\multicolumn{{2}}{{c}}{{{obs_baseline[dv]}}}")
lines.append(r" \# Obs.                    &" + " & ".join(obs_cells_A) + r"\\")
lines.append(r"\midrule")

# =======================
# Panel B: clim1 (Cold, 27°C) and clim2 (Hot, 32°C)
# =======================
lines.append(r"\multicolumn{7}{l}{\textbf{Panel B:} Effect by Hot and Cold Climate Regions}\\")
lines.append(
    r"                    &\multicolumn{1}{c}{(Cold)}&\multicolumn{1}{c}{(Hot)}"
    r"&\multicolumn{1}{c}{(Cold)}&\multicolumn{1}{c}{(Hot)}"
    r"&\multicolumn{1}{c}{(Cold)}&\multicolumn{1}{c}{(Hot)}\\"
)

# Dictionaries for convenience
clim1_dfs = {     # Cold region = 27°C
    "Labour Conflict": Lead_labour_27,
    "Protests": Lead_protest_27,
    "Strikes": Lead_strike_27,
}

clim2_dfs = {     # Hot region = 32°C
    "Labour Conflict": Lead_labour_32,
    "Protests": Lead_protest_32,
    "Strikes": Lead_strike_32,
}

# ------------------------------
# 27°C (Cold region only) rows
# ------------------------------

# 27°C, previous 30 days (Cold only)
coef_27_prev = []
se_27_prev = []

for dv in dv_order:
    df_cold = clim1_dfs[dv]

    est_cold = float(df_cold.loc[0, "est"])
    se_cold  = float(df_cold.loc[0, "se"])

    # Cold cell: value, Hot cell: blank
    coef_27_prev.append(fmt_coef(est_cold, se_cold))  # Cold
    coef_27_prev.append("")                           # Hot blank

    se_27_prev.append(fmt_se(se_cold))                # Cold
    se_27_prev.append("")                             # Hot blank

lines.append(r"Heat Day $\geq 27^{\circ}$C (previous 30 days) &" + " & ".join(coef_27_prev) + r"\\")
lines.append(r"                                            &" + " & ".join(se_27_prev) + r"\\")

# 27°C, next 30 days (Cold only)
coef_27_next = []
se_27_next = []

for dv in dv_order:
    df_cold = clim1_dfs[dv]

    est_cold = float(df_cold.loc[1, "est"])
    se_cold  = float(df_cold.loc[1, "se"])

    coef_27_next.append(fmt_coef(est_cold, se_cold))  # Cold
    coef_27_next.append("")                           # Hot blank

    se_27_next.append(fmt_se(se_cold))                # Cold
    se_27_next.append("")                             # Hot blank

lines.append(r"Heat Day $\geq 27^{\circ}$C (next 30 days)     &" + " & ".join(coef_27_next) + r"\\")
lines.append(r"                                            &" + " & ".join(se_27_next) + r"\\")


# ------------------------------
# 32°C (Hot region only) rows
# ------------------------------

# 32°C, previous 30 days (Hot only)
coef_32_prev = []
se_32_prev = []

for dv in dv_order:
    df_hot = clim2_dfs[dv]

    est_hot = float(df_hot.loc[0, "est"])
    se_hot  = float(df_hot.loc[0, "se"])

    # Cold cell: blank, Hot cell: value
    coef_32_prev.append("")                            # Cold blank
    coef_32_prev.append(fmt_coef(est_hot, se_hot))     # Hot

    se_32_prev.append("")                              # Cold blank
    se_32_prev.append(fmt_se(se_hot))                  # Hot

lines.append(r"Heat Day $\geq 32^{\circ}$C (previous 30 days) &" + " & ".join(coef_32_prev) + r"\\")
lines.append(r"                                            &" + " & ".join(se_32_prev) + r"\\")

# 32°C, next 30 days (Hot only)
coef_32_next = []
se_32_next = []

for dv in dv_order:
    df_hot = clim2_dfs[dv]

    est_hot = float(df_hot.loc[1, "est"])
    se_hot  = float(df_hot.loc[1, "se"])

    coef_32_next.append("")                            # Cold blank
    coef_32_next.append(fmt_coef(est_hot, se_hot))     # Hot

    se_32_next.append("")                              # Cold blank
    se_32_next.append(fmt_se(se_hot))                  # Hot

lines.append(r"Heat Day $\geq 32^{\circ}$C (next 30 days)     &" + " & ".join(coef_32_next) + r"\\")
lines.append(r"                                            &" + " & ".join(se_32_next) + r"\\")


# ------------------------------
# Observations row
# ------------------------------
lines.append(r"\midrule")
obs_cells_B = []
for dv in dv_order:
    obs_cells_B.append(str(obs_clim1[dv]))  # Cold
    obs_cells_B.append(str(obs_clim2[dv]))  # Hot

lines.append(r" \# Obs.                                        &" + " & ".join(obs_cells_B) + r"\\")
lines.append(
    r"\midrule "
    r"County, Month, Year FE & \multicolumn{2}{c}{\checkmark}"
    r"& \multicolumn{2}{c}{\checkmark} & \multicolumn{2}{c}{\checkmark} \\"
    r"Con. Temp.              & \multicolumn{2}{c}{\checkmark}"
    r"& \multicolumn{2}{c}{\checkmark} & \multicolumn{2}{c}{\checkmark} \\"
)
lines.append(r"\bottomrule \addlinespace")
lines.append(r"\end{tabular}")

latex_table = "\n".join(lines)

# Write to .tex file (same name as before if you like)
out_path = Path(directory) / "table_results_abs.tex"
out_path.write_text(latex_table, encoding="utf-8")
print(f"Wrote LaTeX table to {out_path.resolve()}")



Wrote LaTeX table to C:\Users\20234503\Desktop\Research\Weather and Strikes\Project\Results_2\table_results_abs.tex
